In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import re

In [ ]:
# ==================================================
# Sliding windows
# ==================================================
def make_sliding_windows(y, history, horizon, step=1):
    y = np.asarray(y)
    n = len(y)

    X, Y, origins = [], [], []
    for t in range(history, n - horizon + 1, step):
        X.append(y[t-history:t])
        Y.append(y[t:t+horizon])
        origins.append(t)

    return np.array(X), np.array(Y), np.array(origins)


In [ ]:
# ==================================================
# Time-ordered train / test split
# ==================================================
def split_train_test_windows(X, Y, origins, test_ratio=0.2):
    n = len(X)
    split = int(n * (1 - test_ratio))

    return (
        X[:split], Y[:split],
        X[split:], Y[split:],
        origins[:split], origins[split:]
    )


In [ ]:

# ==================================================
# Quantile (pinball) loss
# ==================================================
def quantile_loss(y_pred, y_true, q):
    error = y_true - y_pred
    return torch.mean(torch.maximum(q * error, (q - 1) * error))


In [ ]:
class QuantileMLP(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        output_size,
        min_width=0.01,
        dropout=0.2,
    ):
        super().__init__()

        # ✅ Deeper backbone (more layers)
        self.backbone = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Dropout(dropout),
            nn.LeakyReLU(),

            nn.Linear(hidden_size, hidden_size),
            nn.Dropout(dropout),
            nn.LeakyReLU(),

            nn.Linear(hidden_size, hidden_size),
            nn.Dropout(dropout),
            nn.LeakyReLU(),

            nn.Linear(hidden_size, hidden_size),
            nn.Dropout(dropout),
            nn.LeakyReLU(),

            nn.Linear(hidden_size, hidden_size),
            nn.LeakyReLU(),
        )

        self.lower_head = nn.Linear(hidden_size, output_size)
        self.upper_head = nn.Linear(hidden_size, output_size)

        self.min_width = min_width

    def forward(self, x):
        h = self.backbone(x)

        lower = self.lower_head(h)
        upper_raw = self.upper_head(h)

        # ✅ ensure monotonic + minimum width
        width = torch.cumsum(torch.relu(upper_raw), dim=1)

        # ✅ enforce minimum width (important fix)
        width = width + self.min_width

        upper = lower + width

        return lower, upper

In [ ]:
# ==================================================
# Metrics
# ==================================================
def evaluate_forecasts(results):
    y_true, y_pred, lower, upper = [], [], [], []

    for r in results:
        y_true.extend(r["y_true"])
        y_pred.extend(r["y_pred"])
        lower.extend(r["lower"])
        upper.extend(r["upper"])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    lower = np.array(lower)
    upper = np.array(upper)

    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    coverage = float(np.mean((y_true >= lower) & (y_true <= upper)))

    return rmse, coverage


def evaluate_by_horizon(results):
    H = len(results[0]["y_true"])
    rmse_h, coverage_h = [], []

    for h in range(H):
        yt, yp, lo, up = [], [], [], []
        for r in results:
            yt.append(r["y_true"][h])
            yp.append(r["y_pred"][h])
            lo.append(r["lower"][h])
            up.append(r["upper"][h])

        yt, yp, lo, up = map(np.array, [yt, yp, lo, up])
        rmse_h.append(float(np.sqrt(np.mean((yt - yp) ** 2))))
        coverage_h.append(float(np.mean((yt >= lo) & (yt <= up))))

    return rmse_h, coverage_h


def compute_winkler(results, alpha=0.1):
    scores = []
    for r in results:
        for y, l, u in zip(r["y_true"], r["lower"], r["upper"]):
            width = u - l
            if y < l:
                scores.append(width + 2 / alpha * (l - y))
            elif y > u:
                scores.append(width + 2 / alpha * (y - u))
            else:
                scores.append(width)
    return float(np.mean(scores))



In [ ]:

# ==================================================
# Per-sample RMSE
# ==================================================
def compute_sample_rmse(results):
    errors = []
    for i, r in enumerate(results):
        y_true = np.array(r["y_true"])
        y_pred = np.array(r["y_pred"])
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        errors.append((i, rmse))
    return sorted(errors, key=lambda x: x[1])



In [ ]:

# ==================================================
# Forecast plots
# ==================================================
def plot_single_forecast(results, sample_id=0, title="Neural Quantile"):
    r = results[sample_id]

    history = r["history"]
    y_true = r["y_true"]
    y_pred = r["y_pred"]
    lower = r["lower"]
    upper = r["upper"]

    full_true = np.concatenate([history, y_true])
    t_full = np.arange(len(full_true))
    t_future = np.arange(len(history), len(history) + len(y_true))

    plt.figure(figsize=(10, 4))
    plt.plot(t_full, full_true, label="True", color="black")
    plt.plot(t_future, y_pred, label="Forecast", color="blue")
    plt.fill_between(t_future, lower, upper, alpha=0.3)
    plt.axvline(len(history) - 1, linestyle="--", color="gray")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_best_worst_forecasts_nn(results, dataset_name, n_show=3):
    errors = compute_sample_rmse(results)

    best = errors[:n_show]
    worst = errors[-n_show:]

    print(f"\n📈 Best {n_show} Neural forecasts for {dataset_name}")
    for sid, err in best:
        print(f"Sample {sid}, RMSE={err:.3f}")
        plot_single_forecast(results, sid, "Neural Quantile (Best)")

    print(f"\n📉 Worst {n_show} Neural forecasts for {dataset_name}")
    for sid, err in worst:
        print(f"Sample {sid}, RMSE={err:.3f}")
        plot_single_forecast(results, sid, "Neural Quantile (Worst)")



In [ ]:

# ==================================================
# Neural Quantile Pipeline
# ==================================================
def run_neural_quantile_pipeline(y, test_ratio=0.2, n_epochs=1000):
    history = min(100, len(y) // 3)
    horizon = min(28, len(y) // 6)

    if len(y) < history + horizon + 1:
        return None

    X, Y, _ = make_sliding_windows(y, history, horizon)
    X_train, Y_train, X_test, Y_test, _, _ = split_train_test_windows(
        X, Y, np.arange(len(X)), test_ratio
    )

    if len(X_train) == 0 or len(X_test) == 0:
        return None

#    X_train_t = torch.tensor(X_train, dtype=torch.float32)
#    Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
#    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    
    
    
    
    # ✅ Normalize based on TRAINING DATA ONLY
    mean = X_train.mean()
    std = X_train.std() + 1e-8

    X_train = (X_train - mean) / std
    Y_train = (Y_train - mean) / std
    X_test = (X_test - mean) / std
    Y_test = (Y_test - mean) / std

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)







#    model = QuantileMLP(history, 256, horizon)
#    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model = QuantileMLP(history, 256, horizon)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4   # ✅ regularization
    )

#    model.train()
#    for _ in range(n_epochs):
#        lower, upper = model(X_train_t)
#        loss = (
#            quantile_loss(lower, Y_train_t, 0.1)
#            + quantile_loss(upper, Y_train_t, 0.9)
#        )
#        optimizer.zero_grad()
#        loss.backward()
#        optimizer.step()
        
        
    model.train()
    for _ in range(n_epochs):
        lower, upper = model(X_train_t)

#        loss = (
#            quantile_loss(lower, Y_train_t, 0.1)
#            + quantile_loss(upper, Y_train_t, 0.9)
 #       )
        
        
        
        # ✅ width penalty (ADD THIS)
        width_penalty = torch.mean(upper - lower)

        loss = (
            quantile_loss(lower, Y_train_t, 0.1)
            + quantile_loss(upper, Y_train_t, 0.9)
            + 0.01 * width_penalty   # ✅ discourages overly wide bands
        )

        optimizer.zero_grad()
        loss.backward()

        # ✅ ADD THIS HERE (before optimizer.step)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()   
        
        

    model.eval()
    with torch.no_grad():
        lo, up = model(X_test_t)

    lo, up = lo.numpy(), up.numpy()
    y_pred = (lo + up) / 2

    results = []
    for i in range(len(X_test)):
        results.append({
            "history": X_test[i],
            "y_true": Y_test[i],
            "y_pred": y_pred[i],
            "lower": lo[i],
            "upper": up[i],
        })

    
    # After building `results`
    rmse, coverage = evaluate_forecasts(results)
    rmse_h, coverage_h = evaluate_by_horizon(results)
    winkler = compute_winkler(results)

    return {
        "results_test": results,
        "rmse": rmse,
        "coverage": coverage,
        "winkler": winkler,
        "rmse_h": rmse_h,
        "coverage_h": coverage_h,   # ✅ ADD THIS
}




In [ ]:

# ==================================================
# Helper: parse dataset key
# ==================================================
def parse_key(key):
    parts = key.split("_")
    meta = {"generator": parts[0], "obs_scale": "latent", "sample_size": None}
    for p in parts:
        if p.startswith("n") and p[1:].isdigit():
            meta["sample_size"] = int(p[1:])
        if p.startswith("obs"):
            meta["obs_scale"] = p
    return meta

In [ ]:
def plot_coverage_vs_horizon_nn(
    coverage_h,
    dataset_name,
    target=0.9,
):
    """
    Plot coverage vs forecast horizon for Neural Quantile model.
    """
    horizons = range(1, len(coverage_h) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(horizons, coverage_h, marker="o", label="Neural Coverage")

    plt.axhline(
        target,
        linestyle="--",
        color="red",
        label=f"Target {int(target * 100)}%"
    )

    plt.xlabel("Forecast Horizon")
    plt.ylabel("Coverage")
    plt.title(f"Neural Coverage vs Forecast Horizon\n({dataset_name})")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
print(out.keys())

In [ ]:
print("Winkler value:", out["winkler"])

In [ ]:

# ==================================================
# RUN NEURAL MODEL
# ==================================================
data = np.load("/synthetic_time_series.npz")
nn_summary = []

    
for key in data.files:
    print(f"\nRunning Neural Quantile Model on {key}")

    y = data[key]
    out = run_neural_quantile_pipeline(y)

    if out is None:
        print(f"Skipping {key} (too short)")
        continue
    
    print(f"{key} → Winkler: {out['winkler']}")
    
    # ---- Best / Worst 3 forecast plots ----
    plot_best_worst_forecasts_nn(
        out["results_test"],
        dataset_name=key,
        n_show=3
    )

    # ---- Coverage vs Forecast Horizon (NEW) ----
    plot_coverage_vs_horizon_nn(
        out["coverage_h"],
        dataset_name=key,
        target=0.9
    )

    # ---- Summary ----
    meta = parse_key(key)
    nn_summary.append({
        "dataset": key,
        "generator": meta["generator"],
        "obs_scale": meta["obs_scale"],
        "sample_size": meta["sample_size"],
        "RMSE": out["rmse"],
        "Coverage": out["coverage"],
        "Winkler": out["winkler"],
    })

# df_nn = pd.DataFrame(nn_summary)
# print("\n✅ Neural Network Results Summary")
# print(df_nn)

df_nn = pd.DataFrame(nn_summary)

print("\n================ FINAL RESULTS =================\n")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(df_nn)
print("\nWinkler column:\n", df_nn["Winkler"])
